# Load JSON and Metadata filter JSON field or array

在本笔记本中，我们将使用 Kaggle 上的 IMDB 数据，该数据以原始 JSON 或 CSV 格式提供。我们先将其加载到 Milvus 向量数据库中，然后通过元数据过滤器进行搜索。

开始吧！

In [1]:
import sys,os,time,pprint,json
import numpy as np
import pandas as pd
from IPython.display import display

sys.path.append("..")
import milvus_utilities as _utils

# Decide if you want to read JSON or CSV.
DATA_TYPE = "JSON"

In [2]:
CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')

## Read JSON data into a pandas dataframe

The JSON data comes from https://www.kaggle.com/datasets/nelepie/imdb-genre-classification

In [3]:
if DATA_TYPE=="JSON":

    # Read some JSON data
    df=pd.read_json('data/parsed_data.json')
    print(f"df shape: {df.shape}")
    df=df.T

    # Save top 100 rows as tiny file
    temp=df.head(100).copy()
    temp.to_json('data/tiny_parsed_data.json')
    df=pd.read_json('data/tiny_parsed_data.json')

    # Reset index and cell it movie_index
    df=df.reset_index().rename(columns={'index':'movie_index'})

    # Drop release date, it hsa too many nulls
    df.drop(columns=['releaseDate'], inplace=True)

    # Reverse columns names label, geners
    df=df.rename(columns={'labels':'Genres'})
    df=df.rename(columns={'genre':'labels'})

    # Convert year to a number
    df['film_year']=df.film_year.astype(int)

    # Concatenate连接 Title and Description into 'text' columns
    df['text']=df['title']+' '+df['description']

    # Verify data is valid json
    try:
        # Convert temp to a JSON string and try to parse it
        json.loads(df.to_json())
        print('data is valid JSON')
    except json.JSONDecodeError:
        print('data is not valid JSON')

    print(f"df shape: {df.shape}")
    print(df.dtypes)
    display(df.head(2))

    # Inspect text
    print(f"Example text length: {len(df['text'][0])}")
    print(f"Example text: {df['text'][0]}")

df shape: (7, 10357)
data is valid JSON
df shape: (100, 8)
movie_index       str
title             str
description       str
poster_url        str
labels            str
Genres         object
film_year       int64
text              str
dtype: object


,movie_index,title,description,poster_url,labels,Genres,film_year,text
0,tt6443346,Black Adam,"Nearly 5,000 years after he was bestowed with ...",https://m.media-amazon.com/images/M/MV5BYzZkOG...,SuperHero,"[Action, Adventure, Fantasy]",2022,"Black Adam Nearly 5,000 years after he was bes..."
1,tt10954600,Ant-Man and the Wasp: Quantumania,"Scott Lang and Hope Van Dyne, along with Hank ...",https://m.media-amazon.com/images/M/MV5BNDgyNG...,SuperHero,"[Action, Adventure, Comedy]",2023,Ant-Man and the Wasp: Quantumania Scott Lang a...


Example text length: 240
Example text: Black Adam Nearly 5,000 years after he was bestowed with the almighty powers of the Egyptian gods - and imprisoned just as quickly - Black Adam is freed from his earthly tomb, ready to unleash his unique form of justice on the modern world.


## Read CSV data into a pandas dataframe

本笔记本中使用的数据来自[Kaggle4.8万部电影数据集](https://www.kaggle.com/datasets/yashgupta24/48000-movies-dataset)，除了原始评论文本外，还包含大量元数据。

通常需要进行数据清洗，例如将空字符串替换为""，或将异常或为空的字段替换为中位数值。下面我将直接删除包含空值的行。

In [4]:
# DATA_TYPE="CSV"

In [5]:
if DATA_TYPE=="CSV":
    # Read CSV data
    df=pd.read_csv('data/original_data.csv')

    # Concatenate 'Name', 'Keywards' and 'Description' into 'text' columns
    df['text']=df['Name']+' '+df['Description']+' '+df['ReviewBody']

    # Convert genres from string with commas in it to list of strings
    # 将包含逗号的字符串格式的类型转换为字符串列表
    df['Genres']=df['Genres'].str.split(',')

    # Convert actors from string with commas in it to list of strings.
    df['Actors'] = df['Actors'].str.split(',')

    # Convert keywords from string with commas in it to list of strings.
    df['Keywords'] = df['Keywords'].str.split(',')

    # Extract out just the yeat from the date
    df['DatePublished']=pd.to_datetime(df['DatePublished'],errors='coerce').dt.year
    df['DatePublished']=df['DatePublished'].astype('Int64')

    # Drop extra rating columns
    df.drop(columns=['RatingCount','BestRating','WorstRating'], inplace=True)

    # Inspect text
    print(f"Example text length {len(df.text[0])}")
    pprint.pprint(f"Example text: {df.text[0][:150]}")

    print(df.dtypes)
    display(df.head(2))

## Start up a Zilliz free tier cluster.

本笔记本中的代码使用 Ziliz Cloud 上的完全托管 Milvus 免费试用版。

1. 在创建集合时，请选择默认的“入门”选项，然后为集合命名并创建集群和集合。
2. 在集群主页面上，复制您的 `API 密钥`，并将其保存到本地的 .env 变量中。请参见下方说明如何操作。
3. 同时，在集群主页面上，`复制公共端点 URI`。

💡 注意：为了保护您的令牌安全，最佳实践是使用**环境变量**。请参阅如何将 API 密钥保存为环境变量。

👉🏼 在 Jupyter 中，您需要一个名为 .env 的文件（与笔记本位于同一目录），其中包含如下内容：

- ZILLIZ_API_KEY=f370c...
- OPENAI_API_KEY=sk-H...
- VARIABLE_NAME=value...

## Connect using Milvus Lite

### STEP 1. CONNECT

In [6]:
from pymilvus import MilvusClient
mc=MilvusClient('../milvus_demo.db')

# Check if the server is ready and get collection name
print(f"Type of server: {mc.get_server_version()}")

Type of server: milvus_lite-3.1.1


## Load the Embedding Model checkpoint and use it to create vector embeddings

嵌入模型：我们将使用 HuggingFace 上提供的开源句子转换模型来编码文档文本。我们会从 HuggingFace 下载该模型，并在本地运行。

💡提示：选择句子转换模型的好方法是查看 [MTEB 领先榜单](https://huggingface.co/spaces/mteb/leaderboard)。按“检索平均”列降序排列，选择表现最佳的小型模型。

以下两个模型参数值得注意：

1. EMBEDDING_DIM 指的是嵌入向量的维度或长度。在此情况下，输入文本中每个标记生成的嵌入向量长度相同，均为 1024。这种嵌入大小通常与基于 BERT 的模型相关，其嵌入用于下游任务，如分类、问答或文本生成。

2. MAX_SEQ_LENGTH 是编码器模型可处理的最大上下文长度。在此情况下，如果输入序列超过 512 个标记，所有过长的部分将被（静默地！）截断。因此，需要采用分块策略，将输入文本分割成适合模型输入长度的块。

### STEP 2. DOWNLOAD AN OPEN SOURCE EMBEDDING MODEL

In [7]:
import torch
from sentence_transformers import SentenceTransformer

# Initialize torch settings
torch.backends.cudnn.deterministic=True
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"DEVICE: {DEVICE}")

# Load the model from huggingface model hub
model_name="WhereIsAI/UAE-Large-V1"
encoder=SentenceTransformer(model_name, device=DEVICE)
print(type(encoder))
print(encoder)

# Get the model params and save for later
EMBEDDING_DIM=encoder.get_embedding_dimension()
MAX_SEQ_LENGTH_IN_TOKENS=encoder.get_max_seq_length()
# # Assume tokens are 3 characters long.
# MAX_SEQ_LENGTH = MAX_SEQ_LENGTH_IN_TOKENS * 3
# HF_EOS_TOKEN_LENGTH = 1 * 3
# Test with 512 sequence length.
MAX_SEQ_LENGTH=MAX_SEQ_LENGTH_IN_TOKENS
HF_EOS_TOKEN_LENGTH=1

# Inspect model params
print(f"model name: {model_name}")
print(f"EMBEDDING_DIM: {EMBEDDING_DIM}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")

DEVICE: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

<class 'sentence_transformers.sentence_transformer.model.SentenceTransformer'>
SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 1024, 'pooling_mode': 'cls', 'include_prompt': True})
)
model name: WhereIsAI/UAE-Large-V1
EMBEDDING_DIM: 1024
MAX_SEQ_LENGTH: 512


## Create a Milvus collection

在 Milvus 中，你可以将集合类比为 SQL 数据库中的“表”。该集合将包含以下内容：

- **模型架构**（或无架构的 Milvus 客户端）

    💡 你需要从嵌入模型中获取向量的 EMBEDDING_DIM 参数。常见取值如下：
    - sbert 嵌入模型：1024
    - ada-002 OpenAI 嵌入模型：1536

- **向量索引**，用于高效向量搜索
- **向量距离度量**，用于计算最近邻向量
- **一致性级别**：Milvus 支持事务一致性，但根据 CAP 定理，必须牺牲一定的延迟。💡 由于电影评论搜索并非关键任务，因此`eventually`在此处是可接受的。

# Add a Vector Index

向量索引用于确定在用户提交查询时，查找数据中与该查询最接近的向量所采用的向量搜索算法。

大多数向量索引根据数据库的使用场景（插入向量或搜索向量）而采用不同的参数组合：

- **插入向量**（创建模式）
- **搜索向量**（搜索模式）

请向下滚动文档页面，查看 Milvus 提供的不同向量索引列表。例如：

- FLAT — 确定性穷尽搜索
- IVF_FLAT 或 IVF_SQ8 — 哈希索引（随机近似搜索）
- HNSW — 图形索引（随机近似搜索）
- AUTOINDEX — 根据 OSS 与 Zilliz 云、GPU 类型及数据规模自动确定

除了搜索算法外，我们还需要指定**距离度量**，即定义向量空间中“接近”的标准。在下方单元格中选择了 `HNSW` 搜索索引。其可用的距离度量包括：

- L2 — L2 范数
- IP — 点积
- COSINE — 角度距离

💡 大多数应用场景更适合使用归一化嵌入（normalized embeddings），此时 L2 不适用（每个向量长度为1），IP 和 COSINE 相等。仅当您计划保持嵌入未归一化时，才应选择 L2。

### STEP 3. CREATE A NO-SCHEMA MILVUS COLLECTION AND DEFINE THE DATABASE INDEX.

In [8]:
# Set the Milvus collection name
COLLECTION_NAME='imdb_metadata'

# Add custom HNSW search index to the collection
    # M = max number graph connections per layer. Large M = denser graph.
    # Choice of M: 4~64, larger M for larger data and larger embedding lengths.
M=16
    # efConstruction = num_candidate_nearest_neighbors per layer.
    # Use Rule of thumb: int. 8~512, efConstruction = M * 2.
efConstruction=M*2

# Create the search index for local Milvus server
INDEX_PRARMS=dict({
    'M':M,
    "efConstruction":efConstruction,
})
index_params={
    "index_type":"HNSW",
    "metric_type":"COSINE",
    "params":INDEX_PRARMS,
}

# Use no-schema Milvus client uses flexible json key:value format
# Check if collection already exists, if so drop it.
has=mc.has_collection(COLLECTION_NAME)
if has:
    drop_result=mc.drop_collection(COLLECTION_NAME)
    print(f"Successfully dropped collection: {drop_result}")

# Create the collection
mc.create_collection(
    COLLECTION_NAME,
    EMBEDDING_DIM,
    consistency_level="Eventually",
    auto_id=True,
    overwrite=True,
    params=index_params,
)

print(f"Successfully created collection: `{COLLECTION_NAME}`")
pprint.pprint(mc.describe_collection(COLLECTION_NAME))

Successfully dropped collection: None
Successfully created collection: `imdb_metadata`
{'aliases': [],
 'auto_id': True,
 'collection_id': 0,
 'collection_name': 'imdb_metadata',
 'consistency_level': 0,
 'consistency_level_name': 'Strong',
 'description': '',
 'enable_dynamic_field': True,
 'enable_namespace': False,
 'fields': [{'auto_id': True,
             'description': '',
             'field_id': 0,
             'is_primary': True,
             'name': 'id',
             'params': {},
             'type': <DataType.INT64: 5>},
            {'description': '',
             'field_id': 0,
             'name': 'vector',
             'params': {'dim': 1024},
             'type': <DataType.FLOAT_VECTOR: 101>}],
 'functions': [],
 'num_partitions': 1,
 'num_shards': 1,
 'properties': {}}


# Chunking

在嵌入之前，需要先确定您的分块策略、分块大小和分块重叠度。本节采用以下设置：

- **策略** = 简单的固定分块长度
- **分块大小** = 使用嵌入模型参数 MAX_SEQ_LENGTH
- **重叠度** = 一般建议 10%-15%
- **函数** = Langchain 的`RecursiveCharacterTextSplitter`，用于递归拆分长评论

### STEP 4. PREPARE DATA: CHUNK AND EMBED

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def recursive_splitter_wrapper(text,chunk_size):
    # Default chunk overlap is 10% chunk_size
    chunk_overlap=np.round(chunk_size*0.1,0)

    # Use langchain's convenient recursive chunking method
    text_spliiter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )
    chunks: list[str]=text_spliiter.split_text(text)

    # Replace special characters with spaces
    chunks=[text.replace("<br /><br />", " ") for text in chunks]

    return chunks

# Use recursive splitter to chunk text
def imdb_chunk_text(batch_size,df,chunk_size):

    batch=df.head(batch_size).copy()
    print(f"chunk size: {chunk_size}")
    print(f"original shape: {batch.shape}")

    start_time=time.time()

    # 1. Chunk the text review into chunk_size
    batch['chunk']=batch['text'].apply(recursive_splitter_wrapper,chunk_size=chunk_size)
        # Explode the 'chunk' columns to create new rows for each chunk
    batch=batch.explode('chunk',ignore_index=True)
    print(f"new shape: {batch.shape}")

    # 2. Add embeddings as new columns in df
    embeddings=torch.tensor(encoder.encode(batch['chunk'].tolist()))
        # Normalize the embeddings
    norm=np.linalg.norm(embeddings, axis=1, keepdims=True)
    embeddings=np.asarray(embeddings/norm)

    # 3. Convert embeddings to list of `numpy.ndarray`, each containing `numpy.float32` numbers
    converted_values=list(map(np.float32,embeddings))
    batch['vector']=converted_values

    end_time=time.time()
    print(f"Chunking + embeddings time for {batch_size} docs: {end_time-start_time} sec")

    # Inspect the batch of data
    assert len(batch.chunk[0])<=MAX_SEQ_LENGTH-1
    assert len(batch.vector[0])==EMBEDDING_DIM
    print(f"type embeddings: {type(batch.vector)} of {type(batch.vector[0])}")
    print(f"of numbers: {type(batch.vector[0][0])}")

    return batch

## Chunk and Embed Text Data

In [10]:
# Use the embedding model params
    # chunk_size = MAX_SEQ_LENGTH - HF_EOS_TOKEN_LENGTH
chunk_size=512
chunk_overlap=np.round(chunk_size*0.1,0)

# Chunk a batch of data from pandas DataFrame and inspect it
BATCH_SIZE=100
batch=imdb_chunk_text(BATCH_SIZE,df,chunk_size)
display(batch.head(2))

# Drop the original text columns, keep the new 'chunk' column
batch.drop(columns=['text'], inplace=True)


chunk size: 512
original shape: (100, 8)
new shape: (100, 9)
Chunking + embeddings time for 100 docs: 2.19307017326355 sec
type embeddings: <class 'pandas.Series'> of <class 'numpy.ndarray'>
of numbers: <class 'numpy.float32'>


C:\Users\Administrator\AppData\Local\Temp\ipykernel_8036\780665991.py:38: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  embeddings=np.asarray(embeddings/norm)


,movie_index,title,description,poster_url,labels,Genres,film_year,text,chunk,vector
0,tt6443346,Black Adam,"Nearly 5,000 years after he was bestowed with ...",https://m.media-amazon.com/images/M/MV5BYzZkOG...,SuperHero,"[Action, Adventure, Fantasy]",2022,"Black Adam Nearly 5,000 years after he was bes...","Black Adam Nearly 5,000 years after he was bes...","[0.03977596, 0.017392287, -0.010373332, 0.0086..."
1,tt10954600,Ant-Man and the Wasp: Quantumania,"Scott Lang and Hope Van Dyne, along with Hank ...",https://m.media-amazon.com/images/M/MV5BNDgyNG...,SuperHero,"[Action, Adventure, Comedy]",2023,Ant-Man and the Wasp: Quantumania Scott Lang a...,Ant-Man and the Wasp: Quantumania Scott Lang a...,"[0.08375614, -0.02057303, 0.013680871, 0.00983..."


## Insert data into Milvus

对于每个原始文本片段，我们将把四元组（`vector, text, source, h1, h2`）写入数据库。

**Milvus 客户端封装器仅能处理从字典列表中加载数据。**

否则，Milvus 通常支持从以下格式加载数据：

- pandas 数据框
- 字典列表

下面我们将使用 HuggingFace 提供的嵌入模型，下载其检查点，并在本地运行以作为编码器。

### STEP 5. INSERT CHUNKS AND EMBEDDINGS IN Milvus Lite

In [11]:
# Convert the DataFrame to a list of dictionaries
chunk_list=batch.to_dict(orient='records')

# Insert data into the Milvus collection
print("Strat inserting entities")
start_time=time.time()
insert_result=mc.insert(
    COLLECTION_NAME,
    data=chunk_list,
    progress_bar=True,
)
elaped_time=time.time()-start_time
print(f"Milvus Client insert time for {batch.shape[0]} vectors: {elaped_time} sec")

# Milvus Client does an automatic flush, save data to S3.

# CSV: Milvus Client insert time for 456 vectors: 8.96166205406189 seconds
# JSON: Milvus Client insert time for 100 vectors: 0.3610990047454834 seconds

Strat inserting entities
Milvus Client insert time for 100 vectors: 0.046999454498291016 sec


In [12]:
# Print a single row
chunk_list[0]

{'movie_index': 'tt6443346',
 'title': 'Black Adam',
 'description': 'Nearly 5,000 years after he was bestowed with the almighty powers of the Egyptian gods - and imprisoned just as quickly - Black Adam is freed from his earthly tomb, ready to unleash his unique form of justice on the modern world.',
 'poster_url': 'https://m.media-amazon.com/images/M/MV5BYzZkOGUwMzMtMTgyNS00YjFlLTg5NzYtZTE3Y2E5YTA5NWIyXkEyXkFqcGdeQXVyMjkwOTAyMDU@._V1_QL75_UX190_CR0,0,190,281_.jpg',
 'labels': 'SuperHero',
 'Genres': ['Action', 'Adventure', 'Fantasy'],
 'film_year': 2022,
 'chunk': 'Black Adam Nearly 5,000 years after he was bestowed with the almighty powers of the Egyptian gods - and imprisoned just as quickly - Black Adam is freed from his earthly tomb, ready to unleash his unique form of justice on the modern world.',
 'vector': array([ 0.03977596,  0.01739229, -0.01037333, ..., -0.05583202,
        -0.02902553,  0.03842308], shape=(1024,), dtype=float32)}

## Example PyMilvus utility API calls.

In [13]:
# # 计数行时，会先调用 .flush()。
# 此 API 调用不被 Milvus 客户端支持。
# print(f"计数行：{mc.num_entities(COLLECTION_NAME)}")

In [14]:
# View collection info, 会先调用 .flush()。
start_time=time.time()
pprint.pprint(mc.describe_collection(COLLECTION_NAME))
end_time=time.time()
print(f"timing: {end_time-start_time} sec")
print()

{'aliases': [],
 'auto_id': True,
 'collection_id': 0,
 'collection_name': 'imdb_metadata',
 'consistency_level': 0,
 'consistency_level_name': 'Strong',
 'description': '',
 'enable_dynamic_field': True,
 'enable_namespace': False,
 'fields': [{'auto_id': True,
             'description': '',
             'field_id': 0,
             'is_primary': True,
             'name': 'id',
             'params': {},
             'type': <DataType.INT64: 5>},
            {'description': '',
             'field_id': 0,
             'name': 'vector',
             'params': {'dim': 1024},
             'type': <DataType.FLOAT_VECTOR: 101>}],
 'functions': [],
 'num_partitions': 1,
 'num_shards': 1,
 'properties': {}}
timing: 0.0016634464263916016 sec



In [15]:
# Count rows without 调用 .flush()。
start_time = time.time()
res=mc.query(
    collection_name=COLLECTION_NAME,
    filter="",
    output_fields=["count(*)"],
)
pprint.pprint(res)
end_time=time.time()
print(f"timing: {end_time-start_time} sec")

data: ["{'count(*)': 100}"], extra_info: {}
timing: 0.03708672523498535 sec


In [16]:
# View rows without incurring call to .flush()
OUTPUT_FILEDS=['movie_index','title','description','poster_url','labels','Genres','film_year','chunk']
res=mc.query(collection_name=COLLECTION_NAME,
             filter="id<3",
             output_fields=OUTPUT_FILEDS,)
pprint.pprint(res)

data: ["{'id': 1, 'movie_index': 'tt6443346', 'title': 'Black Adam', 'description': 'Nearly 5,000 years after he was bestowed with the almighty powers of the Egyptian gods - and imprisoned just as quickly - Black Adam is freed from his earthly tomb, ready to unleash his unique form of justice on the modern world.', 'poster_url': 'https://m.media-amazon.com/images/M/MV5BYzZkOGUwMzMtMTgyNS00YjFlLTg5NzYtZTE3Y2E5YTA5NWIyXkEyXkFqcGdeQXVyMjkwOTAyMDU@._V1_QL75_UX190_CR0,0,190,281_.jpg', 'labels': 'SuperHero', 'Genres': ['Action', 'Adventure', 'Fantasy'], 'film_year': 2022, 'chunk': 'Black Adam Nearly 5,000 years after he was bestowed with the almighty powers of the Egyptian gods - and imprisoned just as quickly - Black Adam is freed from his earthly tomb, ready to unleash his unique form of justice on the modern world.'}", "{'id': 2, 'movie_index': 'tt10954600', 'title': 'Ant-Man and the Wasp: Quantumania', 'description': 'Scott Lang and Hope Van Dyne, along with Hank Pym and Janet Van Dyne, 

In [17]:
# Define metadata fields you can filter on
OUTPUT_FILEDS=list(df.columns)
OUTPUT_FILEDS[OUTPUT_FILEDS.index('text')]='chunk'
OUTPUT_FILEDS

['movie_index',
 'title',
 'description',
 'poster_url',
 'labels',
 'Genres',
 'film_year',
 'chunk']

In [18]:
# List distinct genres
GENRES=list(set([genre for genres in df['Genres'] for genre in genres]))
GENRES

['Crime',
 'Thriller',
 'Animation',
 'Horror',
 'Drama',
 'Comedy',
 'Mystery',
 'Adventure',
 'Action',
 'Sci-Fi',
 'Fantasy']

In [19]:
if DATA_TYPE=="CSV":
    # Plot histogram of rating values
    import matplotlib.pyplot as plt
    plt.figure(figsize=(4,2))
    df['RatingValue'].hist()


# Ask a question about your data

在本演示笔记本中，到目前为止：

1. 您的自定义数据已被映射到向量嵌入空间中。
2. 这些向量嵌入已保存至向量数据库中。
接下来，您可以针对您的自定义数据提出问题！

💡 在LLM词汇中：
> **查询**是用户提问的统称。
> 一个查询可以包含多个独立问题，最多可达1000个不同的问题！

> **问题**通常指单个用户提问。
> 在下面的例子中，用户的提问是：“Milvus Client中的AUTOINDEX是什么？”

> **语义搜索** = 针对整个知识库进行快速搜索，找出与用户查询最接近的TOP_K文档片段。

💡 为了保持一致性，所有嵌入数据和查询都应使用相同的模型。

In [20]:
# Define a sample question about your data.

# These 2 questions are for the CSV dataset.
Question1="I'm a medical doctor of Lou Gehrig's disease."
Question2="Bollywood"

# This question for the JSON dataset.
Question3="Dystopia science fiction with a robot."

# Inspect the length of the data
QUERY_LENGTH=len(Question1)
print(f"query length: {QUERY_LENGTH}")

query length: 45


In [21]:
# SELECT A PARTICULAR QUESTION TO ASK.

SAMPLE_QUESTION = Question3

## Execute a vector search

使用 PyMilvus API 进行 Milvus 搜索。

💡 向量搜索本质上是“语义”搜索。例如，如果你搜索“leaky faucet漏水的水龙头”：

> **传统关键词搜索**——无论是“leaky”还是“faucet”，或者两者都必须与文本匹配，才能返回网页或文档链接。

> **语义搜索**——结果还会包含“drippy”、“taps”等词，因为这些词虽然不同，但含义相同。

In [22]:
def mc_run_search(question,filter_expression,top_k):
    # Embed the question using the same encoder
    query_embeddings=_utils.embed_query(encoder,[question])

    # Return top k result with HNSW index
    SEARCH_PARAMS=dict({
        # Re-use index param for num_candidate_nearest_neighbors.
        "ef":INDEX_PRARMS['efConstruction']
    })

    # Run semantic vector search using your query and the vector database
    results=mc.search(
        COLLECTION_NAME,
        data=query_embeddings,
        search_params=SEARCH_PARAMS,
        output_fields=OUTPUT_FILEDS,
        # Milvus can utilize metadata in boolean expressions to filter search.
        filter=filter_expression,
        limit=top_k,
        consistency_level="Eventually"
    )

    # Assemble retrieved context and context metadata.
    # The search result is in the variable `results[0]`, which is type 'pymilvus.orm.search.SearchResult'.
    METADATA_FIELDS=[f for f in OUTPUT_FILEDS if f !='chunk']
    formatted_results,context,context_metadata=_utils.client_assemble_retrieved_context(results,METADATA_FIELDS,top_k)

    return formatted_results,context,context_metadata

In [23]:
# Run the search
if DATA_TYPE=="CSV":
    # Metadata filters for CSV dataset
    expression="RatingValue >= 7"
else:
    # Metadata filters for JSON dataset.
    # expression="film_year >=2019"
    expression='json_contains(Genres,"Sci-Fi") and film_year <2019'
print(f"filter: {expression}")

start_time=time.time()
formatted_results,context,context_metadata=mc_run_search(SAMPLE_QUESTION,expression,2)
elapsed_time=time.time()-start_time
print(f"Milvus Client search time for {len(chunk_list)} vectors: {elapsed_time} seconds")

# Inspect search result
print(f"type: {type(formatted_results)}, count: {len(formatted_results)}")

filter: json_contains(Genres,"Sci-Fi") and film_year <2019
Milvus Client search time for 100 vectors: 0.15930604934692383 seconds
type: <class 'list'>, count: 2


In [25]:
# Display poster link
from IPython.display import Image

# Loop throught recommended movies, display poster, print metadata
seen_movies=[]
for i in range(len(context)):
    print(f"Retrieved result #{i+1}")
    print(f"distance = {formatted_results[i][0]}")

    # Get the movie_index
    movie_index=context_metadata[i]['movie_index']
    print(f"movie index: {movie_index}")

    # Don't display the same movie_index twice
    if movie_index in seen_movies:
        continue
    else:
        seen_movies.append(movie_index)

        # Display the first poster link as a rendered渲染 image
        if DATA_TYPE=="CSV":
            poster='PosterLink'
        else:
            poster='poster_url'
        x=Image(url=context_metadata[i][poster],width=150,height=200)
        display(x)

        # Print the rest of the movie info
        pprint.pprint(f"Chunk text: {context[i]}")

        # print metadata except the movie_index and poster link
        for key,value in context_metadata[i].items():
            if key!=poster or key!='movie_index':
                print(f"{key}: {value}")
        print()

Retrieved result #1
distance = 0.580215334892273
movie index: tt0434409


('Chunk text: V for Vendetta In a future British dystopian society, a shadowy '
 'freedom fighter, known only by the alias of "V", plots to overthrow the '
 'tyrannical government - with the help of a young woman.')
movie_index: tt0434409
title: V for Vendetta
description: In a future British dystopian society, a shadowy freedom fighter, known only by the alias of "V", plots to overthrow the tyrannical government - with the help of a young woman.
poster_url: https://m.media-amazon.com/images/M/MV5BOTI5ODc3NzExNV5BMl5BanBnXkFtZTcwNzYxNzQzMw@@._V1_QL75_UX190_CR0,0,190,281_.jpg
labels: SuperHero
Genres: ['Action', 'Drama', 'Sci-Fi']
film_year: 2005

Retrieved result #2
distance = 0.5398455858230591
movie index: tt0489099


('Chunk text: Jumper A teenager with teleportation abilities suddenly finds '
 'himself in the middle of an ancient war between those like him and their '
 'sworn annihilators.')
movie_index: tt0489099
title: Jumper
description: A teenager with teleportation abilities suddenly finds himself in the middle of an ancient war between those like him and their sworn annihilators.
poster_url: https://m.media-amazon.com/images/M/MV5BMjEwOTkyOTI3M15BMl5BanBnXkFtZTcwNTQxMjU1MQ@@._V1_QL75_UX190_CR0,0,190,281_.jpg
labels: SuperHero
Genres: ['Action', 'Adventure', 'Sci-Fi']
film_year: 2008



In [26]:
# Drop collection
mc.drop_collection(COLLECTION_NAME)